# Computer Vision — Activity 1: Dissecting Faster R-CNN (Two-Stage)

> **Colab Setup Reminder:** Before running the code cells, select **Runtime → Change runtime type → T4 GPU**.

---

## Part A — Hand Calculations

### A1. Number of Anchors
* **(a) Original Faster R-CNN (VGG-16 without FPN, stride 16, 9 anchors/position, 800 × 608 image):**
  * Feature map size: `(800 / 16) × (608 / 16)` = **50 × 38** (`1,900` grid positions)
  * Total anchors: `50 × 38 × 9` = **17,100**

* **(b) Torchvision Faster R-CNN (ResNet-50 + FPN, 5 levels, 3 anchors/position):**
  * Formula from the 5 printed FPN feature maps `(1, 256, H_k, W_k)` in Part B:
    $$\text{Total Anchors} = 3 \times \sum_{k=1}^{5} (H_k \times W_k)$$
  * *Example (for a 3:2 image resized so shorter side = 800 px → `800 × 1200`):*
    * Level `0` (stride 4): `200 × 300 = 60,000` positions → `180,000` anchors
    * Level `1` (stride 8): `100 × 150 = 15,000` positions → `45,000` anchors
    * Level `2` (stride 16): `50 × 75 = 3,750` positions → `11,250` anchors
    * Level `3` (stride 32): `25 × 38 = 950` positions → `2,850` anchors
    * Level `pool` (stride 64): `13 × 19 = 247` positions → `741` anchors
    * **Total:** `79,947 × 3` = **239,841** anchors *(Note: Part B Step 1 prints the exact value for your downloaded `street.jpg`)*.
  * **Why is it much larger than in (a)?**
    1. FPN adds high-resolution feature maps at **stride 4** and **stride 8** to detect small objects (stride 4 has `4 × 4 = 16×` as many spatial positions per unit area as stride 16).
    2. Torchvision resizes the *shorter* side to `800 px` (producing a larger pixel area than `800 × 608`).

---

### A2 & A3. IoU with Ground Truth & RPN Anchor Labels
* **Ground Truth:** `GT = (50, 50, 150, 150)` → `Area(GT) = 100 × 100 = 10,000`
* **RPN Rule:** `IoU ≥ 0.7` → **Positive**, `IoU < 0.3` → **Negative**, `0.3 ≤ IoU < 0.7` → **Ignored**

| Proposal | Box (x1, y1, x2, y2) | Area(P) | Intersection Box with GT | Area(P ∩ GT) | Area(P ∪ GT) | A2: IoU | A3: RPN Label |
| :--- | :--- | :---: | :--- | :---: | :---: | :---: | :---: |
| **P1** | `(60, 60, 160, 160)` | 10,000 | `(60, 60, 150, 150)` | 90 × 90 = 8,100 | 11,900 | `8100 / 11900` ≈ **0.6807** | **Ignored** |
| **P2** | `(55, 45, 145, 155)` | 9,900 | `(55, 50, 145, 150)` | 90 × 100 = 9,000 | 10,900 | `9000 / 10900` ≈ **0.8257** | **Positive** |
| **P3** | `(200, 200, 300, 300)` | 10,000 | No overlap | 0 | 20,000 | `0 / 20000` = **0.0000** | **Negative** |
| **P4** | `(70, 40, 170, 140)` | 10,000 | `(70, 50, 150, 140)` | 80 × 90 = 7,200 | 12,800 | `7200 / 12800` = **0.5625** | **Ignored** |

---

### A4. Non-Maximum Suppression (NMS Threshold = 0.5)
1. **Sort by confidence score:** `P1 (0.90)` > `P2 (0.85)` > `P3 (0.80)` > `P4 (0.60)`
2. **Keep `P1 (60, 60, 160, 160)`**, compare pairwise IoU with remaining boxes:
   * **P1 vs P2:** Intersect `(60, 60, 145, 155)` → Area = `85 × 95 = 8,075`, Union = `10,000 + 9,900 - 8,075 = 11,825` → IoU = `8075 / 11825` ≈ **0.6829** (`> 0.5` → **Discard P2**)
   * **P1 vs P3:** No overlap → IoU = **0.0000** (`≤ 0.5` → **Keep P3**)
   * **P1 vs P4:** Intersect `(70, 60, 160, 140)` → Area = `90 × 80 = 7,200`, Union = `10,000 + 10,000 - 7,200 = 12,800` → IoU = `7200 / 12800` = **0.5625** (`> 0.5` → **Discard P4**)
3. **Select next remaining box `P3 (200, 200, 300, 300)`** (no other boxes left).
4. **Remaining boxes:** **P1** and **P3**.
5. **Does the box with the best GT IoU survive?** **No.** `P2` had the highest IoU with GT (`0.8257`), but during inference NMS has no ground truth and ranks purely by confidence score (`P1` 0.90 > `P2` 0.85). Because `IoU(P1, P2) = 0.6829 > 0.5`, `P1` suppresses `P2`.

---

### A5. Bounding Box Regression
* **Given:** `(xa, ya) = (100, 100)`, `(wa, ha) = (128, 128)`, `t = (0.10, -0.20, 0.2231, 0)`
* **Predicted center & size:**
  * `x = 100 + 0.10 × 128` = **112.8**
  * `y = 100 + (-0.20) × 128` = **74.4**
  * `w = 128 × exp(0.2231) ≈ 128 × 1.25` = **160.0**
  * `h = 128 × exp(0) = 128 × 1` = **128.0**
* **Convert to corner coordinates `(x ± w/2, y ± h/2)`:**
  * `x1 = 112.8 - 80` = **32.8**
  * `y1 = 74.4 - 64` = **10.4**
  * `x2 = 112.8 + 80` = **192.8**
  * `y2 = 74.4 + 64` = **138.4**
* **Final Predicted Box:** **`(32.8, 10.4, 192.8, 138.4)`**


---
## Part B — Each Stage in Code

### Setup & Test Image Download

In [ ]:
import os
import time
import urllib.request
import torch
import matplotlib.pyplot as plt
from torchvision.io import read_image, ImageReadMode
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.utils import draw_bounding_boxes

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Download a COCO street scene image (with fallback) if street.jpg does not exist
if not os.path.exists("street.jpg"):
    urls = [
        "http://images.cocodataset.org/val2017/000000001584.jpg",
        "https://ultralytics.com/images/bus.jpg",
    ]
    for url in urls:
        try:
            urllib.request.urlretrieve(url, "street.jpg")
            print(f"Downloaded street.jpg from {url}")
            break
        except Exception as e:
            print(f"Failed {url}: {e}")
else:
    print("Found existing street.jpg")


### Step 1 — Run Faster R-CNN Stage by Stage & Inspect Shapes

In [ ]:
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights).eval().to(device)

img = read_image("street.jpg", mode=ImageReadMode.RGB)
x = weights.transforms()(img).to(device)

with torch.no_grad():
    images, _ = model.transform([x])                                        # resize (shorter side 800 px) + normalization
    features = model.backbone(images.tensors)                               # backbone + FPN: 5 feature maps
    proposals, _ = model.rpn(images, features)                              # STAGE 1: RPN (+ NMS 0.7)
    dets, _ = model.roi_heads(features, proposals, images.image_sizes)      # STAGE 2 (+ per-class NMS 0.5)
    dets = model.transform.postprocess(dets, images.image_sizes, [x.shape[-2:]])

print("original size (height, width):", tuple(img.shape[-2:]))
print("size after resize (height, width):", images.image_sizes[0])

anchors_per_loc = model.rpn.anchor_generator.num_anchors_per_location()
total_anchors = 0
for (k, v), a in zip(features.items(), anchors_per_loc):
    h, w = v.shape[-2], v.shape[-1]
    level_anchors = h * w * a
    total_anchors += level_anchors
    print(f"FPN level {k:>4}: shape={tuple(v.shape)}, positions={h}x{w}={h*w}, anchors={level_anchors}")

print("anchors per position at each level:", anchors_per_loc)
print("total anchors across all 5 FPN levels (for A1b):", total_anchors)
print("number of RPN proposals:", proposals[0].shape[0])
print("number of final detections:", dets[0]["boxes"].shape[0])

names = weights.meta["categories"]
print("final detections (score > 0.5):", [names[i] for i in dets[0]["labels"][dets[0]["scores"] > 0.5]])


### Step 2 — Draw First 50 RPN Proposals vs. Final Detections (`score > 0.5`)

In [ ]:
orig_h, orig_w = img.shape[-2:]
resized_h, resized_w = images.image_sizes[0]
scale_x = orig_w / resized_w
scale_y = orig_h / resized_h

# Scale proposals from resized coordinates back to original image coordinates
props_50 = proposals[0][:50].detach().cpu().clone()
props_50[:, [0, 2]] *= scale_x
props_50[:, [1, 3]] *= scale_y

# Final detections with score > 0.5
keep = dets[0]["scores"] > 0.5
det_boxes = dets[0]["boxes"][keep].detach().cpu()
det_labels = [
    f"{names[lbl.item()]} {score.item():.2f}"
    for lbl, score in zip(dets[0]["labels"][keep], dets[0]["scores"][keep])
]

img_props = draw_bounding_boxes(img, props_50, colors="yellow", width=2)
img_dets = (
    draw_bounding_boxes(img, det_boxes, labels=det_labels, colors="lime", width=3)
    if len(det_boxes) > 0 else img.clone()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(img_props.permute(1, 2, 0).numpy())
axes[0].set_title(f"Stage 1: First 50 RPN Proposals (of {proposals[0].shape[0]})")
axes[0].axis("off")

axes[1].imshow(img_dets.permute(1, 2, 0).numpy())
axes[1].set_title(f"Stage 2: Final Detections (score > 0.5, count={len(det_boxes)})")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Step 3 — Reduce Number of RPN Proposals (`1000` → `50` → `10`)

In [ ]:
proposal_counts = [1000, 50, 10]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

try:
    for ax, top_n in zip(axes, proposal_counts):
        model.rpn._post_nms_top_n["testing"] = top_n
        with torch.no_grad():
            out = model([x])[0]
        keep = out["scores"] > 0.5
        boxes = out["boxes"][keep].detach().cpu()
        labels_list = [names[i.item()] for i in out["labels"][keep]]
        box_texts = [
            f"{names[l.item()]} {s.item():.2f}"
            for l, s in zip(out["labels"][keep], out["scores"][keep])
        ]

        print(f"top_n={top_n:>4} -> detections (score > 0.5): {len(boxes):>2} | classes: {labels_list}")

        vis = (
            draw_bounding_boxes(img, boxes, labels=box_texts, colors="lime", width=3)
            if len(boxes) > 0 else img.clone()
        )
        ax.imshow(vis.permute(1, 2, 0).numpy())
        ax.set_title(f"RPN top_n = {top_n} ({len(boxes)} detections)")
        ax.axis("off")
finally:
    # Reset back to default (1000)
    model.rpn._post_nms_top_n["testing"] = 1000
    print("Reset model.rpn._post_nms_top_n[testing] =", model.rpn._post_nms_top_n["testing"])

plt.tight_layout()
plt.show()


### Step 4 — Vary Second-Stage NMS Threshold (`0.1`, `0.5`, `0.9`)

In [ ]:
nms_thresholds = [0.1, 0.5, 0.9]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

try:
    for ax, thresh in zip(axes, nms_thresholds):
        model.roi_heads.nms_thresh = thresh
        with torch.no_grad():
            out = model([x])[0]
        keep = out["scores"] > 0.5
        boxes = out["boxes"][keep].detach().cpu()
        labels_list = [names[i.item()] for i in out["labels"][keep]]
        box_texts = [
            f"{names[l.item()]} {s.item():.2f}"
            for l, s in zip(out["labels"][keep], out["scores"][keep])
        ]

        print(f"nms_thresh={thresh:.1f} -> detections (score > 0.5): {len(boxes):>2} | classes: {labels_list}")

        vis = (
            draw_bounding_boxes(img, boxes, labels=box_texts, colors="cyan", width=3)
            if len(boxes) > 0 else img.clone()
        )
        ax.imshow(vis.permute(1, 2, 0).numpy())
        ax.set_title(f"NMS thresh = {thresh} ({len(boxes)} detections)")
        ax.axis("off")
finally:
    # Reset back to default (0.5) before timing
    model.roi_heads.nms_thresh = 0.5
    print("Reset model.roi_heads.nms_thresh =", model.roi_heads.nms_thresh)

plt.tight_layout()
plt.show()


### Step 5 — Measure Faster R-CNN Inference Time

In [ ]:
def measure_time(fn, n=10):
    fn()                                         # warm-up
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n): fn()
    if device == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000 # ms per image

with torch.no_grad():
    print("Faster R-CNN time (ms):", measure_time(lambda: model([x])))


---
## Activity 1 — Reflection Questions

1. **Why is the RPN called the “first stage”? What does the RPN do, and what does it not do yet?**
   * It is called the first stage because it narrows down hundreds of thousands of dense anchor boxes across the FPN feature maps into ~1,000 candidate regions (*proposals*) before any object classification happens.
   * **What it does:** It scores every anchor for class-agnostic *objectness* (object vs. background), predicts initial bounding-box offsets to tighten anchors around potential objects, and applies NMS (threshold `0.7`) to keep the top ~1,000 proposals.
   * **What it does NOT do yet:** It does not know *which class* each object belongs to (e.g., `person`, `bus`, `car`) and does not produce the final refined detections—that is handled by Stage 2 (`roi_heads`).

2. **Based on Step 3, what happens when there are too few proposals?**
   * Reducing proposals from `1000` to `50` and `10` causes **recall to drop sharply**: fewer objects are detected, and smaller, occluded, or lower-objectness objects disappear first.
   * Because Stage 2 only inspects the proposals passed to it by the RPN, any object omitted from the top-`k` proposal list is permanently missed and cannot be recovered.
   * *(After running Step 3 on Colab, note the exact detection counts and which specific objects disappeared in your image.)*

3. **Based on Step 4, explain in your own words what NMS does and the effect of a threshold that is too low or too high.**
   * **What NMS does:** Non-Maximum Suppression removes duplicate boxes for the same class. It keeps the highest-confidence box, discards any other same-class box overlapping it with `IoU > threshold`, and repeats on the remaining boxes.
   * **Threshold too low (`0.1`):** Over-suppression—adjacent or slightly overlapping distinct objects of the same class (e.g., nearby pedestrians or parked cars) get mistakenly deleted, lowering recall.
   * **Threshold too high (`0.9`):** Under-suppression—multiple overlapping predictions around the exact same object usually have IoUs between `0.5` and `0.85`, so they survive and create duplicate boxes on one object.

4. **Why is RoI Align needed before the classification head?**
   * RPN proposals have varying aspect ratios, sizes, and floating-point coordinates, whereas the Stage 2 classification and regression head uses **fully connected layers** that require a fixed-size feature map (`7 × 7`) for every proposal.
   * RoI Align crops each proposal from its FPN feature level and uses **bilinear interpolation** (avoiding the coordinate rounding/quantization misalignment of older RoI Pooling) to resize it cleanly to `7 × 7`.
